# EAA Pose Estimation Pipeline — Google Colab

Pipeline RGB → 3D 25-khớp chuẩn NTU-120 cho PKU-MMD v1/v2 và TSU.

**Yêu cầu**:
- Runtime: GPU (T4 hoặc cao hơn)
- Dataset PKU-MMD v1/v2 hoặc TSU đã có trên Google Drive
- Đã chạy Module 1 (`filter_pku_interactions`) cho PKU-MMD v1

**Thứ tự chạy**:
1. Section 1: Mount Drive + kiểm tra GPU
2. Section 2: Cài đặt dependencies
3. Section 3: Clone / upload repo
4. Section 4: Chọn dataset và cấu hình paths
5. Section 5: (PKU v1 only) Chạy Module 1 filter nếu chưa có
6. Section 6: Chạy Module 2 pose pipeline
7. Section 7: Kiểm tra output

## Section 1 — Mount Google Drive & Kiểm tra GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected. Switch Runtime → Change runtime type → GPU')

## Section 2 — Cài đặt Dependencies

In [ ]:
# Core dependencies
!pip install -q openpyxl PyYAML scipy tqdm opencv-python-headless

# MMPose stack (GPU required)
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q openmim
!mim install mmengine -q
!mim install 'mmcv>=2.1' -q
!mim install 'mmdet>=3.2' -q
!mim install 'mmpose>=1.3' -q
!pip install -q lap  # ByteTrack dependency

## Section 3 — Upload / Clone Repo

In [ ]:
import os, sys

# --- Option A: Repo đã upload lên Drive ---
# REPO_PATH = '/content/drive/MyDrive/DATN/Skeleton-EAA-Pose'

# --- Option B: Upload từ máy local (chạy cell này rồi upload zip) ---
# from google.colab import files
# uploaded = files.upload()   # chọn Skeleton-EAA-Pose.zip
# !unzip -q Skeleton-EAA-Pose.zip -d /content/
# REPO_PATH = '/content/Skeleton-EAA-Pose'

# --- Option C: Clone từ git (nếu đã push lên GitHub) ---
# !git clone https://github.com/<username>/Skeleton-EAA-Pose.git /content/Skeleton-EAA-Pose
# REPO_PATH = '/content/Skeleton-EAA-Pose'

# Đặt REPO_PATH theo option bạn chọn ở trên:
REPO_PATH = '/content/drive/MyDrive/DATN/Skeleton-EAA-Pose'

sys.path.insert(0, REPO_PATH)
os.chdir(REPO_PATH)
print('Repo path:', REPO_PATH)
print('eaa_pose importable:', os.path.isdir(os.path.join(REPO_PATH, 'eaa_pose')))

## Section 4 — Cấu hình Paths

Chỉnh `DATASET` và các path phù hợp với cấu trúc Google Drive của bạn.

In [ ]:
# ── Chọn dataset ──────────────────────────────────────────────────────────────
DATASET = 'pku_v1'   # 'pku_v1' | 'pku_v2' | 'tsu'

DRIVE_ROOT = '/content/drive/MyDrive/DATN'

if DATASET == 'pku_v1':
    CONFIG_FILE  = f'{REPO_PATH}/configs/pku_v1.yaml'
    VIDEO_DIR    = f'{DRIVE_ROOT}/PKU_MMD_v1/RGB'          # thư mục chứa *.avi
    SKELETON_DIR = f'{DRIVE_ROOT}/PKU_MMD_v1/Skeleton'    # thư mục skeleton gốc (cho filter)
    LABEL_DIR    = f'{DRIVE_ROOT}/PKU_MMD_v1/Label'       # label gốc
    ACTIONS_XLSX = f'{DRIVE_ROOT}/PKU_MMD_v1/Actions.xlsx'

    # Output paths (sau Module 1)
    FILTERED_LABEL_DIR    = f'{DRIVE_ROOT}/PKU_MMD_v1/Label_daily'
    FILTERED_ACTIONS_XLSX = f'{DRIVE_ROOT}/PKU_MMD_v1/Actions_daily_v1.xlsx'
    OUT_DIR               = f'{DRIVE_ROOT}/PKU_MMD_v1/samples_npy'

elif DATASET == 'pku_v2':
    CONFIG_FILE  = f'{REPO_PATH}/configs/pku_v2.yaml'
    VIDEO_DIR    = f'{DRIVE_ROOT}/PKU_MMD_v2/RGB'
    LABEL_DIR    = f'{DRIVE_ROOT}/PKU_MMD_v2/Label'
    ACTIONS_XLSX = f'{DRIVE_ROOT}/PKU_MMD_v2/Actions.xlsx'
    FILTERED_LABEL_DIR    = LABEL_DIR    # v2 không cần filter interaction
    FILTERED_ACTIONS_XLSX = ACTIONS_XLSX
    OUT_DIR = f'{DRIVE_ROOT}/PKU_MMD_v2/samples_npy'

elif DATASET == 'tsu':
    CONFIG_FILE  = f'{REPO_PATH}/configs/tsu.yaml'
    VIDEO_DIR    = f'{DRIVE_ROOT}/TSU/mp4'              # thư mục chứa *.mp4
    LABEL_DIR    = f'{DRIVE_ROOT}/TSU/Annotation_v1.0'  # thư mục annotation CSV
    OUT_DIR      = f'{DRIVE_ROOT}/TSU/samples_npy'
    FILTERED_LABEL_DIR    = LABEL_DIR
    FILTERED_ACTIONS_XLSX = None

DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print(f'Dataset: {DATASET}, Device: {DEVICE}')
print(f'Video dir: {VIDEO_DIR}')
print(f'Segments dir: {FILTERED_LABEL_DIR}')
print(f'Output dir: {OUT_DIR}')

## Section 5 — (PKU v1 only) Module 1: Lọc Interaction Actions

Bỏ qua nếu đã chạy và có `Label_daily/` + `Actions_daily_v1.xlsx`.

In [ ]:
import os

if DATASET != 'pku_v1':
    print('Bỏ qua — không phải PKU v1')
elif os.path.isdir(FILTERED_LABEL_DIR) and len(os.listdir(FILTERED_LABEL_DIR)) > 0:
    print(f'Label_daily đã tồn tại ({len(os.listdir(FILTERED_LABEL_DIR))} files) — bỏ qua filter.')
else:
    print('Chạy Module 1 lọc interaction...')
    !python -m eaa_pose.filter_pku_interactions \
        --skeleton-dir  {SKELETON_DIR} \
        --label-dir     {LABEL_DIR} \
        --src-xlsx      {ACTIONS_XLSX} \
        --out-label-dir {FILTERED_LABEL_DIR} \
        --out-xlsx      {FILTERED_ACTIONS_XLSX}

    print('Xong. Số file label sau filter:', len(os.listdir(FILTERED_LABEL_DIR)))

## Section 6 — Module 2: Chạy Pose Pipeline (GPU)

In [ ]:
# Tải model checkpoints (chỉ cần lần đầu)
import subprocess, os

CKPT_DIR = '/content/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

# RTMDet-m person detector
DET_CFG  = 'rtmdet_m_640-8xb32_coco-person'
DET_CKPT = f'{CKPT_DIR}/rtmdet_m_8xb32-100e_coco-obj365-person-235e8209.pth'
if not os.path.exists(DET_CKPT):
    print('Tải RTMDet checkpoint...')
    !mim download mmdet --config {DET_CFG} --dest {CKPT_DIR}

# RTMW3D whole-body 3D pose estimator
POSE_CFG  = 'rtmw3d-l_8xb64_cocktail14-384x288'
POSE_CKPT = f'{CKPT_DIR}/rtmw3d-l_cock14-0d4ad840_20240407.pth'
if not os.path.exists(POSE_CKPT):
    print('Tải RTMW3D checkpoint...')
    !mim download mmpose --config {POSE_CFG} --dest {CKPT_DIR}

print('Checkpoints sẵn sàng.')

In [ ]:
# Kiểm tra config path từ mim download thực tế
import glob
det_cfgs  = glob.glob(f'{CKPT_DIR}/*rtmdet*.py')
pose_cfgs = glob.glob(f'{CKPT_DIR}/*rtmw3d*.py')
det_ckpts  = glob.glob(f'{CKPT_DIR}/*rtmdet*.pth')
pose_ckpts = glob.glob(f'{CKPT_DIR}/*rtmw3d*.pth')

print('Det config:', det_cfgs)
print('Pose config:', pose_cfgs)
print('Det ckpt:', det_ckpts)
print('Pose ckpt:', pose_ckpts)

DET_CFG_PATH  = det_cfgs[0]  if det_cfgs  else ''
DET_CKPT_PATH = det_ckpts[0] if det_ckpts else ''
POSE_CFG_PATH  = pose_cfgs[0]  if pose_cfgs  else ''
POSE_CKPT_PATH = pose_ckpts[0] if pose_ckpts else ''

In [ ]:
# Chạy toàn bộ pose pipeline
extra_args = ''
if DATASET == 'pku_v1':
    extra_args = f'--actions-xlsx "{FILTERED_ACTIONS_XLSX}"'

cmd = f"""
python -m eaa_pose.run_pose \
    --config "{CONFIG_FILE}" \
    --video-dir "{VIDEO_DIR}" \
    --segments-dir "{FILTERED_LABEL_DIR}" \
    --out-dir "{OUT_DIR}" \
    --device {DEVICE} \
    {extra_args}
""".strip()

# Override detector/estimator checkpoints if downloaded via mim
# (Thêm args này nếu cấu hình mặc định trong pku_v1.yaml chưa khớp với ckpt tải về)
# cmd += f' --det-config "{DET_CFG_PATH}" --det-checkpoint "{DET_CKPT_PATH}"'
# cmd += f' --pose-config "{POSE_CFG_PATH}" --pose-checkpoint "{POSE_CKPT_PATH}"'

print('Command:', cmd)
!{cmd}

## Section 7 — Kiểm tra Output

In [ ]:
import numpy as np, os, glob

npy_files = sorted(glob.glob(f'{OUT_DIR}/*.npy'))
print(f'Số file .npy đã lưu: {len(npy_files)}')

if npy_files:
    sample_path = npy_files[0]
    arr = np.load(sample_path)
    print(f'\nFile mẫu: {os.path.basename(sample_path)}')
    print(f'  Shape: {arr.shape}  (T, M, 25, 6)')
    print(f'  Dtype: {arr.dtype}')

    T, M, J, C = arr.shape
    print(f'  Frames: {T}, Persons: {M}, Joints: {J}, Channels: {C}')
    print(f'  valid_mask mean: {arr[..., 4].mean():.3f}  (1=valid, 0=missing)')
    print(f'  reconstructed mean: {arr[..., 5].mean():.3f}')
    print(f'  SpineBase (joint 0) xyz at frame 0: {arr[0, 0, 0, :3]}')
    print(f'  (should be ~[0,0,0] after root-center normalization)')

In [ ]:
# Visualize skeleton trajectory of first sample
import matplotlib.pyplot as plt
import numpy as np

if npy_files:
    arr = np.load(npy_files[0])  # (T, M, 25, 6)
    T = arr.shape[0]

    # NTU-120 bone connections (0-based)
    BONES = [
        (0,1),(1,20),(20,2),(2,3),          # spine + neck + head
        (20,4),(4,5),(5,6),(6,7),(7,21),(7,22),   # left arm
        (20,8),(8,9),(9,10),(10,11),(11,23),(11,24), # right arm
        (0,12),(12,13),(13,14),(14,15),      # left leg
        (0,16),(16,17),(17,18),(18,19),      # right leg
    ]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax_idx, t_idx in enumerate([0, T//2, T-1]):
        ax = axes[ax_idx]
        joints = arr[t_idx, 0, :, :3]  # (25, 3) — x, y, z
        valid  = arr[t_idx, 0, :, 4]   # (25,)

        for i, j in BONES:
            if valid[i] > 0.5 and valid[j] > 0.5:
                ax.plot([joints[i, 0], joints[j, 0]],
                        [-joints[i, 1], -joints[j, 1]],  # flip y for display
                        'b-', linewidth=1.5)

        ax.scatter(joints[valid > 0.5, 0], -joints[valid > 0.5, 1],
                   c='red', s=20, zorder=5)
        ax.set_title(f'Frame {t_idx}')
        ax.set_aspect('equal')
        ax.axis('off')

    plt.suptitle(f'{os.path.basename(npy_files[0])} — skeleton visualization')
    plt.tight_layout()
    plt.show()